# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. **`1 1.3333333333333333`**. Один пропуск (`None` стал `NaN`); после `fillna(0)` среднее равно `(1 + 0 + 3) / 3 = 4/3`.
2. **`2`**. Пробелы исчезают после `strip()`, регистр выравнивается через `lower()`; остаются значения `a` и `b`.
3. **`3`**. `drop_duplicates()` возвращает новую таблицу, но переменная `df` не изменена, потому что нет присваивания.
4. **`1005`**. `.str.replace(" ", "")` превращает `"1 000"` в `"1000"`, а `pd.to_numeric` позволяет сложить 1000 и 5.
5. **`[9, 10]`**. `.dt.month` извлекает месяцы из дат: сентябрь и октябрь. Метод `.tolist()` здесь взят непосредственно из кода вопроса преподавателя.


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
import pandas as pd

# Справочник из урока 6: одинаковые города могли быть записаны по-разному.
CITY = {"алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда", "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print("Прочитано:", len(df), "строк")

    # Суммы: сначала убираем пробелы, затем переводим текст в числа.
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("После очистки сумм:", len(df), "строк; NaN:", df["amount"].isna().sum())

    # Города: убираем пробелы, приводим к нижнему регистру, заменяем по словарю.
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("После очистки городов:", len(df), "строк; NaN:", df["city"].isna().sum(),
          "; городов:", df["city"].nunique())

    # Важно присвоить результат: drop_duplicates не меняет df без присваивания.
    df = df.drop_duplicates()
    print("После удаления дублей:", len(df), "строк; order_id уникален:", df["order_id"].is_unique)

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("После перевода дат:", len(df), "строк; тип даты:", df["order_date"].dtype)

    df = df[df["customer_id"] != "TEST"].copy()
    print("После удаления TEST:", len(df), "строк")
    print("Итоговые проверки: amount NaN =", df["amount"].isna().sum(),
          "; city NaN =", df["city"].isna().sum(),
          "; order_id уникален =", df["order_id"].is_unique)
    return df

df_clean = clean("orders.csv")
print("Менеджеру: для честной аналитики осталось", len(df_clean),
      "заказов; отменённые не удаляли, чтобы оценить долю отмен.")


Прочитано: 5463 строк
После очистки сумм: 5463 строк; NaN: 0
После очистки городов: 5463 строк; NaN: 0 ; городов: 6
После удаления дублей: 5283 строк; order_id уникален: True
После перевода дат: 5283 строк; тип даты: datetime64[ns]
После удаления TEST: 5280 строк
Итоговые проверки: amount NaN = 0 ; city NaN = 0 ; order_id уникален = True
Менеджеру: для честной аналитики осталось 5280 заказов; отменённые не удаляли, чтобы оценить долю отмен.


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
# Считаем выручку только оплаченных заказов; отмены остаются в таблице.
paid = df_clean["status"] == "оплачен"
sep = df_clean["month"] == 9
octo = df_clean["month"] == 10
sep_rev = df_clean.loc[paid & sep, "amount"].sum()
oct_rev = df_clean.loc[paid & octo, "amount"].sum()
print("Сентябрь:", sep_rev, "₸; октябрь:", oct_rev, "₸")
print("Изменение: {:.2f}%".format((oct_rev / sep_rev - 1) * 100))

worst_city = ""
worst_change = 0
for city_name in df_clean["city"].unique():
    here = df_clean["city"] == city_name
    s = df_clean.loc[paid & here & sep, "amount"].sum()
    o = df_clean.loc[paid & here & octo, "amount"].sum()
    change = (o / s - 1) * 100
    print(city_name, "| сентябрь:", s, "| октябрь:", o,
          "| изменение: {:.2f}%".format(change))
    if change < worst_change:
        worst_change = change
        worst_city = city_name

print("Менеджеру: падение соответствует уроку 4 (около −2,3%).")
print("Самое сильное относительное падение:", worst_city,
      "({:.2f}%).".format(worst_change))


Сентябрь: 35243410.0 ₸; октябрь: 34419520.0 ₸
Изменение: -2.34%
Алматы | сентябрь: 14501070.0 | октябрь: 14643020.0 | изменение: 0.98%
Астана | сентябрь: 9034220.0 | октябрь: 8850530.0 | изменение: -2.03%
Караганда | сентябрь: 3648520.0 | октябрь: 3371680.0 | изменение: -7.59%
Шымкент | сентябрь: 4558650.0 | октябрь: 4044010.0 | изменение: -11.29%
Атырау | сентябрь: 923990.0 | октябрь: 969860.0 | изменение: 4.96%
Актобе | сентябрь: 2576960.0 | октябрь: 2540420.0 | изменение: -1.42%
Менеджеру: падение соответствует уроку 4 (около −2,3%).
Самое сильное относительное падение: Шымкент (-11.29%).


## [База] 3. Журнал очистки для менеджера

| Шаг | Убрано строк на шаге | Строк осталось | Изменение выручки октябрь / сентябрь |
|---|---:|---:|---:|
| Исходный расчёт: только оплаченные | 0 | 5 463 | −22,74% |
| Очистили суммы и названия городов | 0 | 5 463 | −22,74% |
| Удалили 180 дублей | 180 | 5 283 | −19,59% |
| Исключили 3 тестовых заказа `TEST` | 3 | **5 280** | **−2,34%** |

**Менеджеру:** сильнее всего картину искажают три тестовых заказа на 9 999 990 ₸ каждый: без них падение почти исчезает. Дубли также завышали сентябрь, поэтому их нужно удалять до расчётов.


In [4]:
# Пересчитываем журнал на тех же данных, чтобы проверить числа таблицы.
original = pd.read_csv("orders.csv")
original["amount"] = pd.to_numeric(original["amount"].str.replace(" ", ""))

# Даты здесь пока текст: из урока можно брать первые 7 символов (год-месяц).
def revenue_change(df):
    paid_rows = df["status"] == "оплачен"
    sept_rows = df["order_date"].str[:7] == "2026-09"
    oct_rows = df["order_date"].str[:7] == "2026-10"
    sept_sum = df.loc[paid_rows & sept_rows, "amount"].sum()
    oct_sum = df.loc[paid_rows & oct_rows, "amount"].sum()
    return (oct_sum / sept_sum - 1) * 100

without_duplicates = original.drop_duplicates()
without_test = without_duplicates[without_duplicates["customer_id"] != "TEST"]
print("Исходный расчёт:", len(original), "строк; {:.2f}%".format(revenue_change(original)))
print("Без дублей:", len(without_duplicates), "строк; {:.2f}%".format(revenue_change(without_duplicates)))
print("Без TEST:", len(without_test), "строк; {:.2f}%".format(revenue_change(without_test)))
print("Менеджеру: главная ошибка исходного отчёта — учёт заказов TEST.")


Исходный расчёт: 5463 строк; -22.74%
Без дублей: 5283 строк; -19.59%
Без TEST: 5280 строк; -2.34%
Менеджеру: главная ошибка исходного отчёта — учёт заказов TEST.


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
# После очистки пропуски остаются только в delivery_days.
missing = df_clean["delivery_days"].isna()
print("Пропусков сроков доставки:", missing.sum())
print("По городам:")
print(df_clean.loc[missing, "city"].value_counts())
print("По статусам:")
print(df_clean.loc[missing, "status"].value_counts())

sh_days = df_clean.loc[df_clean["city"] == "Шымкент", "delivery_days"]
rate_dropna = (sh_days.dropna() <= 2).mean() * 100
rate_median = (sh_days.fillna(sh_days.median()) <= 2).mean() * 100
rate_zero = (sh_days.fillna(0) <= 2).mean() * 100
print("Шымкент, доставка ≤2 дня: dropna = {:.2f}%, fillna(медиана) = {:.2f}%, fillna(0) = {:.2f}%".format(
    rate_dropna, rate_median, rate_zero))
print("Менеджеру: показываем долю по известным срокам (dropna).")
print("Медиана подставляет выдуманные сроки, а 0 ошибочно считает пропуски быстрой доставкой.")


Пропусков сроков доставки: 158
По городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
По статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64
Шымкент, доставка ≤2 дня: dropna = 29.84%, fillna(медиана) = 28.99%, fillna(0) = 31.83%
Менеджеру: показываем долю по известным срокам (dropna).
Медиана подставляет выдуманные сроки, а 0 ошибочно считает пропуски быстрой доставкой.


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [6]:
# В ошибочном коде было два недостатка:
# 1) drop_duplicates() не был присвоен обратно переменной df.
# 2) lower() не удаляет пробелы и не переводит латиницу в названия на кириллице.
df_ai = pd.read_csv("orders.csv")
df_ai = df_ai.drop_duplicates()
df_ai["city"] = df_ai["city"].str.strip().str.lower().map(CITY)
print("Разных городов:", df_ai["city"].nunique())
print("Строк после удаления дублей:", len(df_ai))
print("Пропусков в городах:", df_ai["city"].isna().sum())
print("Менеджеру: исправленный код даёт 6 городов и 5 283 строки; TEST здесь ещё не удалён.")


Разных городов: 6
Строк после удаления дублей: 5283
Пропусков в городах: 0
Менеджеру: исправленный код даёт 6 городов и 5 283 строки; TEST здесь ещё не удалён.


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [7]:
weekdays = df_clean["order_date"].dt.day_name().value_counts()
print("Число заказов по дням недели:")
print(weekdays)
smallest = weekdays.min()
largest = weekdays.max()
difference = (largest - smallest) / largest * 100
print("День с минимумом:")
print(weekdays[weekdays == smallest])
print("День с максимумом:")
print(weekdays[weekdays == largest])
print("В самый спокойный день заказов меньше на {:.2f}%".format(difference))
print("Менеджеру: планируйте больше курьеров на загруженные дни (например, четверг),")
print("а на понедельник можно предусмотреть меньшую смену при прочих равных.")


Число заказов по дням недели:
order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
День с минимумом:
order_date
Monday    684
Name: count, dtype: int64
День с максимумом:
order_date
Thursday    786
Name: count, dtype: int64
В самый спокойный день заказов меньше на 12.98%
Менеджеру: планируйте больше курьеров на загруженные дни (например, четверг),
а на понедельник можно предусмотреть меньшую смену при прочих равных.


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [8]:
sh = df_clean[df_clean["city"] == "Шымкент"]
sh_sep = sh[sh["month"] == 9]
sh_oct = sh[sh["month"] == 10]
print("Шымкент: средняя доставка в сентябре = {:.2f} дня, в октябре = {:.2f} дня".format(
    sh_sep["delivery_days"].mean(), sh_oct["delivery_days"].mean()))

# Ищем последний быстрый и первый медленный заказ после сортировки по дате.
fast = sh[sh["delivery_days"] < 5].sort_values("order_date")
slow = sh[sh["delivery_days"] >= 5].sort_values("order_date")
print("Последний заказ со сроком <5 дней:")
print(fast[["order_date", "delivery_days"]].tail(1))
print("Первый заказ со сроком ≥5 дней:")
print(slow[["order_date", "delivery_days"]].head(1))
first_slow = slow.iloc[0]["order_date"]
print("Быстрых доставок после первого медленного заказа:",
      len(sh[(sh["order_date"] >= first_slow) & (sh["delivery_days"] < 5)]))

others = df_clean[df_clean["city"] != "Шымкент"]
print("Другие 5 городов: сентябрь = {:.2f}, октябрь = {:.2f} дня".format(
    others.loc[others["month"] == 9, "delivery_days"].mean(),
    others.loc[others["month"] == 10, "delivery_days"].mean()))
print("Менеджеру: сбой начинается 1 октября 2026 года примерно в 10:00 в Шымкенте.")
print("В первую очередь проверить курьеров, склад и подрядчика в этом городе.")


Шымкент: средняя доставка в сентябре = 2.03 дня, в октябре = 6.54 дня
Последний заказ со сроком <5 дней:
              order_date  delivery_days
2780 2026-09-30 22:00:00            2.0
Первый заказ со сроком ≥5 дней:
              order_date  delivery_days
2793 2026-10-01 10:00:00            5.0
Быстрых доставок после первого медленного заказа: 0
Другие 5 городов: сентябрь = 2.00, октябрь = 2.01 дня
Менеджеру: сбой начинается 1 октября 2026 года примерно в 10:00 в Шымкенте.
В первую очередь проверить курьеров, склад и подрядчика в этом городе.


## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [9]:
def to_amount(s):
    # В уроке 6 разбирали str.replace и pd.to_numeric.
    # Первый replace убирает пробелы, второй меняет запятую на точку.
    # errors="coerce" превращает некорректные значения в NaN вместо ошибки;
    # после этого обязательно считаем, сколько таких пропусков появилось.
    return pd.to_numeric(s.str.replace(" ", "").str.replace(",", "."), errors="coerce")

example = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print("Проверка на четырёх строках:")
print(to_amount(example))
print("Некорректных значений в примере:", to_amount(example).isna().sum())

amounts = to_amount(pd.read_csv("orders.csv")["amount"])
print("Пропусков после перевода amount из файла:", amounts.isna().sum())
print("Сумма всех заказов:", amounts.sum(), "₸")
print("Менеджеру: форматы суммы можно привести к одному виду, но пропуски надо проверять,")
print("иначе неверные исходные записи незаметно уменьшат выручку.")


Проверка на четырёх строках:
0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64
Некорректных значений в примере: 1
Пропусков после перевода amount из файла: 0
Сумма всех заказов: 109883360.0 ₸
Менеджеру: форматы суммы можно привести к одному виду, но пропуски надо проверять,
иначе неверные исходные записи незаметно уменьшат выручку.
